In [ ]:
import subprocess, sys

def pip(args):
    subprocess.run([sys.executable, '-m', 'pip'] + args,
                   check=True, capture_output=True)

pip(['install', '-q', 'transformers>=4.49.0', '--upgrade'])
pip(['install', '-q', 'qwen-vl-utils', 'torchvision'])
pip(['install', '-q', 'packaging', 'datasets'])
pip(['install', '-q', 'rouge_score', 'evaluate', 'underthesea', 'bert-score', 'nltk'])
pip(['uninstall', 'numpy', '-y'])
pip(['install', '-q', 'numpy==1.26.4'])

import transformers, numpy as np
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')

In [ ]:
from pathlib import Path
import torch

KAGGLE_WORKING         = Path('/kaggle/working')
SRC_DATASET            = '/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset'
DST_DATASET            = str(KAGGLE_WORKING / 'vi_chart_dataset')
VIETNAMESE_DATA_PATH   = Path('/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese')
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / 'images'
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / 'viet_chart_vqa.jsonl'

MODEL_DIR      = Path('/kaggle/input/models/maituananh511/qwen2-vl-lora/pytorch/default/1')

CHART_TEST_N   = 500
VIETNAMESE_N   = 200
EVAL_TOTAL     = CHART_TEST_N + VIETNAMESE_N
MAX_NEW_TOKENS = 64
METRICS        = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']

CSV_OUTPUT   = KAGGLE_WORKING / 'eval_qwen2vl_lora.csv'
CHART_OUTPUT = KAGGLE_WORKING / 'eval_qwen2vl_lora_chart.png'

n_gpu = torch.cuda.device_count()
print(f'GPU count: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} — {p.total_memory // 1024**3} GB')
print(f'\nEval target : {EVAL_TOTAL} samples')
print(f'Model dir   : {MODEL_DIR}')

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
    device_map={'': device},
    trust_remote_code=True,
).eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Model loaded — {n_params:.2f}B params')
print(f'VRAM used   : {torch.cuda.memory_allocated(device) / 1024**3:.2f} GB')

In [ ]:
from datasets import load_from_disk
from PIL import Image
import json, shutil, os

def is_dataset_complete(dst, src):
    if not os.path.exists(dst):
        return False
    src_files = {os.path.relpath(os.path.join(r, f), src)
                 for r, _, fs in os.walk(src) for f in fs}
    dst_files = {os.path.relpath(os.path.join(r, f), dst)
                 for r, _, fs in os.walk(dst) for f in fs}
    missing = src_files - dst_files
    if missing:
        print(f'Thiếu {len(missing)} files')
        return False
    return True

if is_dataset_complete(DST_DATASET, SRC_DATASET):
    print('Dataset đã copy đầy đủ, skip.')
else:
    if os.path.exists(DST_DATASET):
        shutil.rmtree(DST_DATASET)
    print('Copying dataset ...')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy xong.')

vi_chart_dataset = load_from_disk(DST_DATASET)
print(vi_chart_dataset)

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))
print(f'Vietnamese records: {len(vietnamese_records)} loaded')

In [ ]:
from PIL import Image

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        return {'role': role, 'content': str(turn.get('content') or turn.get('value', ''))}
    return {'role': 'assistant', 'content': str(turn)}

chart_test_raw   = vi_chart_dataset['test']
chart_n          = min(CHART_TEST_N, len(chart_test_raw))
chart_test_items = [chart_test_raw[i] for i in range(chart_n)]
print(f'vi_chart test   : {chart_n} samples')

vn_test_items = []
for record in reversed(vietnamese_records):
    if len(vn_test_items) >= VIETNAMESE_N:
        break
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'Warning: {img_path}: {e}')
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs)-1, 2)]
    for idx, (q, a) in enumerate(pairs):
        if len(vn_test_items) >= VIETNAMESE_N:
            break
        rid = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_test_items.append({'id': rid, 'image': image, 'conversations': [q, a]})

print(f'vietnamese test  : {len(vn_test_items)} samples')
eval_dataset = chart_test_items + vn_test_items
print(f'Total            : {len(eval_dataset)} samples')

has_img = sum(1 for x in eval_dataset if x.get('image') is not None)
print(f'Samples có ảnh   : {has_img} / {len(eval_dataset)}')

In [ ]:
import os, gc, nltk
import pandas as pd
from qwen_vl_utils import process_vision_info
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
from underthesea import word_tokenize
from tqdm import tqdm
from PIL import Image

nltk_path = '/usr/share/nltk_data'
os.makedirs(nltk_path, exist_ok=True)
nltk.data.path.append(nltk_path)
for pkg in ['punkt', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, download_dir=nltk_path, quiet=True)

SYSTEM_MSG = (
    'Bạn là trợ lý AI thông minh, chuyên phân tích biểu đồ. '
    'Hãy trả lời câu hỏi về biểu đồ trong ảnh bằng tiếng Việt ngắn gọn và chính xác.'
)
rouge_sc = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method1

def get_pil_image(item):
    img = item.get('image')
    if img is None:
        return None
    if isinstance(img, Image.Image):
        return img
    if isinstance(img, dict) and 'bytes' in img:
        import io
        return Image.open(io.BytesIO(img['bytes'])).convert('RGB')
    if isinstance(img, (str, os.PathLike)):
        return Image.open(img).convert('RGB')
    return None

def generate_response(item):
    question = str(item['conversations'][0]['content'])
    pil_img  = get_pil_image(item)

    content = []
    if pil_img is not None:
        content.append({'type': 'image', 'image': pil_img})
    content.append({'type': 'text', 'text': f'Câu hỏi: {question}'})

    messages = [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user',   'content': content},
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    enc = processor(
        text=[text],
        images=image_inputs if image_inputs else None,
        videos=video_inputs if video_inputs else None,
        padding=True,
        return_tensors='pt',
    ).to(device)

    try:
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=(
                    processor.tokenizer.pad_token_id
                    if processor.tokenizer.pad_token_id is not None
                    else processor.tokenizer.eos_token_id
                ),
            )
    except torch.cuda.OutOfMemoryError:
        print(f'  OOM sample {item["id"]} — fallback text-only')
        torch.cuda.empty_cache(); gc.collect()
        msgs_text = [
            {'role': 'system', 'content': SYSTEM_MSG},
            {'role': 'user',   'content': f'Câu hỏi: {question}'},
        ]
        text2 = processor.apply_chat_template(
            msgs_text, tokenize=False, add_generation_prompt=True
        )
        enc = processor(text=[text2], return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=processor.tokenizer.eos_token_id,
            )

    input_len = enc['input_ids'].shape[1]
    response  = processor.tokenizer.decode(
        out[0][input_len:], skip_special_tokens=True
    ).strip()
    torch.cuda.empty_cache(); gc.collect()
    return response

def compute_metrics(gt, response):
    ref = word_tokenize(gt, format='text').split()
    hyp = word_tokenize(response, format='text').split() if response else ['']
    bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
    meteor = float(nltk_meteor([ref], hyp))
    r      = rouge_sc.score(gt, response)
    return {
        'bleu':   bleu,
        'meteor': meteor,
        'rouge1': r['rouge1'].fmeasure,
        'rouge2': r['rouge2'].fmeasure,
        'rougeL': r['rougeL'].fmeasure,
    }

print('Helper functions defined')

In [ ]:
items = [i for i in eval_dataset if all(k in i for k in ['id', 'conversations'])]
print(f'Evaluating {len(items)} samples ...')
print('=' * 60)

results = []
for it in tqdm(items, desc='Eval'):
    response = generate_response(it)
    gt       = str(it['conversations'][1]['content'])
    m        = compute_metrics(gt, response)
    results.append({
        'id':           it['id'],
        'question':     str(it['conversations'][0]['content']),
        'ground_truth': gt,
        'response':     response,
        **m,
    })

df = pd.DataFrame(results)

print('\nComputing BERTScore ...')
try:
    import bert_score as bs_lib
    _, _, F1 = bs_lib.score(
        df['response'].tolist(),
        df['ground_truth'].tolist(),
        lang='vi', verbose=False, rescale_with_baseline=False,
    )
    df['bertscore'] = F1.tolist()
except Exception as e:
    print(f'BERTScore failed: {e}')
    df['bertscore'] = [0.0] * len(df)

avg = {m: df[m].mean() for m in METRICS}

print('\n' + '=' * 60)
print('KẾT QUẢ — Qwen2-VL LoRA Fine-tuned:')
print('=' * 60)
for k, v in avg.items():
    print(f'  {k:<12}: {v:.4f}')

df.to_csv(str(CSV_OUTPUT), index=False, encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

summary_df = pd.DataFrame([{
    'Model': 'Qwen2-VL LoRA Fine-tuned',
    **{m.upper(): round(avg.get(m, 0.0), 4) for m in METRICS}
}]).set_index('Model')

print(f'KẾT QUẢ ({EVAL_TOTAL} samples)\n')
display(summary_df)

summary_df.reset_index().to_csv(
    str(KAGGLE_WORKING / 'eval_qwen2vl_lora_summary.csv'),
    index=False, encoding='utf-8'
)

x    = np.arange(len(METRICS))
vals = [avg.get(m, 0.0) for m in METRICS]
clrs = ['#2ecc71' if v == max(vals) else '#3498db' for v in vals]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, vals, color=clrs, width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in METRICS], fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, min(1.15, max(vals) * 1.3 + 0.05))
ax.set_title(f'Qwen2-VL LoRA Fine-tuned  ({EVAL_TOTAL} samples)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(CHART_OUTPUT), dpi=150)
plt.show()
print(f'Saved: {CHART_OUTPUT}')

In [ ]:
print('Top 10 samples BERTScore')
display(
    df.nlargest(10, 'bertscore')
    [['id', 'question', 'ground_truth', 'response', 'bleu', 'meteor', 'bertscore']]
)

print('\nTop 10 samples BERTScore')
display(
    df.nsmallest(10, 'bertscore')
    [['id', 'question', 'ground_truth', 'response', 'bleu', 'meteor', 'bertscore']]
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, m in enumerate(METRICS):
    ax = axes[i]
    if m in df.columns:
        ax.hist(df[m], bins=30, color='#3498db', alpha=0.8, edgecolor='white')
        ax.axvline(df[m].mean(), color='red', linestyle='--',
                   linewidth=1.5, label=f'mean={df[m].mean():.3f}')
        ax.legend(fontsize=8)
    ax.set_title(m.upper(), fontsize=11)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')

plt.suptitle(
    f'Histogram score — Qwen2-VL LoRA Fine-tuned  ({len(df)} samples)',
    fontsize=13,
)
plt.tight_layout()

hist_path = KAGGLE_WORKING / 'histogram_qwen2vl_lora.png'
plt.savefig(str(hist_path), dpi=150)
plt.show()
print(f'Saved: {hist_path}')

In [ ]:
VINTERN_LORA_CSV = Path('/kaggle/input/datasets/maituananh511/700-vinternlora-result/debug_vintern_lora.csv')

if VINTERN_LORA_CSV.exists():
    df_vintern = pd.read_csv(str(VINTERN_LORA_CSV))
    df_vintern.columns = [c.strip() for c in df_vintern.columns]
    col_map = {c.lower(): c for c in df_vintern.columns}

    avg_vintern = {}
    for m in METRICS:
        actual_col = col_map.get(m.lower())
        avg_vintern[m] = df_vintern[actual_col].mean() if actual_col else 0.0

    print(f'Vintern-LoRA — {len(df_vintern)} samples')

    rows = [
        {'Model': 'Qwen2-VL LoRA Fine-tuned', **{m.upper(): round(avg.get(m, 0.0), 4) for m in METRICS}},
        {'Model': 'Vintern-LoRA',             **{m.upper(): round(avg_vintern.get(m, 0.0), 4) for m in METRICS}},
    ]
    cmp_df = pd.DataFrame(rows).set_index('Model')

    def highlight_best(s):
        return ['background-color: #d4edda; font-weight: bold' if v == s.max() else '' for v in s]
    def highlight_worst(s):
        return ['background-color: #f8d7da' if v == s.min() else '' for v in s]

    print(f'\nSO SÁNH ({EVAL_TOTAL} samples | Xanh = cao nhất | Đỏ = thấp nhất)\n')
    display(cmp_df.style.apply(highlight_best).apply(highlight_worst))

    cmp_df.reset_index().to_csv(
        str(KAGGLE_WORKING / 'eval_lora_vs_vintern_summary.csv'),
        index=False, encoding='utf-8'
    )

    all_models = {'Qwen2-VL LoRA': avg, 'Vintern-LoRA': avg_vintern}
    n      = len(all_models)
    x      = np.arange(len(METRICS))
    width  = 0.8 / n
    colors = ['#27ae60', '#d65f5f']

    fig, ax = plt.subplots(figsize=(13, 5))
    for idx, (mname, avg_m) in enumerate(all_models.items()):
        offset = idx * width - (n - 1) * width / 2
        vals   = [avg_m.get(m, 0.0) for m in METRICS]
        bars   = ax.bar(x + offset, vals, width, label=mname, color=colors[idx])
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in METRICS], fontsize=10)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.2)
    ax.set_title(f'Qwen2-VL LoRA vs Vintern-LoRA  ({EVAL_TOTAL} samples)', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    cmp_chart = KAGGLE_WORKING / 'eval_lora_vs_vintern_chart.png'
    plt.savefig(str(cmp_chart), dpi=150)
    plt.show()
    print(f'Saved: {cmp_chart}')
else:
    print(f'Không tìm thấy {VINTERN_LORA_CSV} — bỏ qua cell so sánh.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

path_qwen = Path('/kaggle/working/eval_qwen2vl_lora.csv')
path_vintern = Path('/kaggle/input/datasets/maituananh511/700-vinternlora-result/debug_vintern_lora.csv')
OUTPUT_FILE = Path('/kaggle/working/comparison_metrics_distribution.png')

df_q = pd.read_csv(str(path_qwen))
df_v = pd.read_csv(str(path_vintern))

df_q.columns = [c.strip().lower() for c in df_q.columns]
df_v.columns = [c.strip().lower() for c in df_v.columns]

METRICS_LIST = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougel', 'bertscore']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, m in enumerate(METRICS_LIST):
    ax = axes[i]
    
    if m in df_q.columns:
        ax.hist(df_q[m], bins=30, color='#3498db', alpha=0.6, 
                edgecolor='white', label='Qwen2-VL LoRA')
        
    if m in df_v.columns:
        ax.hist(df_v[m], bins=30, color='#e74c3c', alpha=0.5, 
                edgecolor='white', label='Vintern-LoRA')
    
    ax.set_title(m.upper(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Sample Count')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.2)

plt.suptitle(f'Score Distribution Comparison: Qwen2-VL vs Vintern-LoRA ({len(df_q)} samples)', 
             fontsize=14, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.savefig(str(OUTPUT_FILE), dpi=150, bbox_inches='tight')
plt.show()

print(f"Export Successful!")
print(f"You can download the file from the right sidebar (Output tab) at: {OUTPUT_FILE}")